# 05 — Final Load Prep & KPI Computation

Compute the KPI framework promised in the problem statement and export the Tableau-ready dataset plus a set of pre-aggregated summary tables used directly in the dashboard.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
CLEAN_PATH         = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_dataset.csv'
TABLEAU_READY_PATH = PROJECT_ROOT / 'data' / 'processed' / 'tableau_ready_dataset.csv'
KPI_DIR            = PROJECT_ROOT / 'data' / 'processed' / 'kpi'
KPI_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CLEAN_PATH, parse_dates=['start_time', 'end_time'])
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')

## KPI framework

| KPI | Definition | Government use |
|---|---|---|
| Severity Index by State | mean(severity) per state | Prioritise state-level interventions |
| Accident Density by City | count per city | Identify hotspot cities |
| Weather Risk Score | % severity ≥ 3 per weather condition | Trigger weather advisories |
| Rush-Hour Severity Rate | mean severity in rush vs non-rush | Staff emergency response |
| Infrastructure Risk Flag | % high-severity per POI type | Prioritise infrastructure upgrades |
| YoY Growth Rate | (year - prev year) / prev year | Track programme effectiveness |
| Nighttime Severity Premium | mean severity night - day | Justify lighting investments |

### KPI 1 — Severity index by state

In [ ]:
state_kpi = (
    df.groupby('state').agg(
        accident_count=('severity', 'size'),
        severity_index=('severity', 'mean'),
        high_severity_rate=('is_high_severity', 'mean'),
        avg_duration_min=('duration_min', 'mean'),
    )
    .round(3)
    .sort_values('severity_index', ascending=False)
)
state_kpi['high_severity_rate'] = (state_kpi['high_severity_rate'] * 100).round(2)
state_kpi.to_csv(KPI_DIR / 'kpi_state.csv')
state_kpi.head(10)

### KPI 2 — Accident density by city (top 100)

In [ ]:
city_kpi = (
    df.groupby(['state', 'city']).agg(
        accident_count=('severity', 'size'),
        severity_index=('severity', 'mean'),
        high_severity_rate=('is_high_severity', 'mean'),
    )
    .round(3)
    .sort_values('accident_count', ascending=False)
    .head(100)
)
city_kpi['high_severity_rate'] = (city_kpi['high_severity_rate'] * 100).round(2)
city_kpi.to_csv(KPI_DIR / 'kpi_city_top100.csv')
city_kpi.head(10)

### KPI 3 — Weather risk score

In [ ]:
weather_kpi = (
    df.groupby('weather_condition').agg(
        accident_count=('severity', 'size'),
        severity_index=('severity', 'mean'),
        weather_risk_score=('is_high_severity', 'mean'),
    )
    .round(3)
)
weather_kpi['weather_risk_score'] = (weather_kpi['weather_risk_score'] * 100).round(2)
weather_kpi = weather_kpi[weather_kpi['accident_count'] >= 1000].sort_values('weather_risk_score', ascending=False)
weather_kpi.to_csv(KPI_DIR / 'kpi_weather.csv')
weather_kpi.head(15)

### KPI 4 — Rush-hour severity rate

In [ ]:
rush_kpi = (
    df.groupby('is_rush_hour').agg(
        accident_count=('severity', 'size'),
        severity_index=('severity', 'mean'),
        high_severity_rate=('is_high_severity', 'mean'),
    )
    .round(3)
)
rush_kpi['high_severity_rate'] = (rush_kpi['high_severity_rate'] * 100).round(2)
rush_kpi.index = rush_kpi.index.map({True: 'rush_hour', False: 'non_rush'})
rush_kpi.to_csv(KPI_DIR / 'kpi_rush_hour.csv')
rush_kpi

### KPI 5 — Infrastructure risk flag

In [ ]:
poi_cols = ['traffic_signal', 'junction', 'crossing', 'stop', 'railway',
            'station', 'amenity', 'bump', 'roundabout', 'give_way', 'no_exit', 'traffic_calming']
poi_cols = [c for c in poi_cols if c in df.columns]

rows = []
for col in poi_cols:
    present = df[df[col]]
    rows.append({
        'poi': col,
        'accident_count': len(present),
        'severity_index': round(present['severity'].mean(), 3) if len(present) else np.nan,
        'high_severity_rate_pct': round(present['is_high_severity'].mean() * 100, 2) if len(present) else np.nan,
    })
poi_kpi = pd.DataFrame(rows).sort_values('high_severity_rate_pct', ascending=False)
poi_kpi.to_csv(KPI_DIR / 'kpi_poi.csv', index=False)
poi_kpi

### KPI 6 — Year-over-year growth rate

In [ ]:
yearly = df.groupby('year').agg(
    accident_count=('severity', 'size'),
    severity_index=('severity', 'mean'),
).round(3)
yearly['yoy_growth_pct'] = (yearly['accident_count'].pct_change() * 100).round(2)
yearly.to_csv(KPI_DIR / 'kpi_yoy.csv')
yearly

### KPI 7 — Nighttime severity premium

In [ ]:
if 'sunrise_sunset' in df.columns:
    daynight = df.groupby('sunrise_sunset').agg(
        accident_count=('severity', 'size'),
        severity_index=('severity', 'mean'),
        high_severity_rate=('is_high_severity', 'mean'),
    ).round(3)
    daynight['high_severity_rate'] = (daynight['high_severity_rate'] * 100).round(2)
    daynight.to_csv(KPI_DIR / 'kpi_day_night.csv')
    print(daynight)
    if {'Day', 'Night'}.issubset(set(daynight.index)):
        premium = daynight.loc['Night', 'severity_index'] - daynight.loc['Day', 'severity_index']
        print(f'\nNighttime severity premium: {premium:+.3f} severity points')

## Tableau-ready export

Re-shape the row-level dataset with only the columns the Tableau workbook actually needs. Trimming unused columns keeps the workbook performant on Tableau Public (which has a 15M-row and 10 GB extract limit).

In [ ]:
tableau_cols = [
    # IDs & geography
    'state', 'city', 'county', 'zipcode', 'timezone',
    'start_lat', 'start_lng',
    # time
    'start_time', 'end_time', 'date', 'year', 'month', 'month_name',
    'day_of_week', 'day_name', 'hour', 'season', 'is_weekend', 'is_rush_hour',
    # severity
    'severity', 'severity_label', 'is_high_severity', 'duration_min', 'distance_mi',
    # weather
    'weather_condition', 'temperature_f', 'humidity', 'visibility_mi',
    'wind_speed_mph', 'precipitation_in', 'pressure_in',
    'sunrise_sunset', 'civil_twilight',
    # infrastructure POI
    'traffic_signal', 'junction', 'crossing', 'stop', 'railway',
    'station', 'amenity', 'bump', 'roundabout', 'give_way', 'no_exit', 'traffic_calming',
]
tableau_cols = [c for c in tableau_cols if c in df.columns]

tableau_df = df[tableau_cols].copy()
print(f'Tableau dataset: {len(tableau_df):,} rows × {tableau_df.shape[1]} columns')

In [ ]:
TABLEAU_READY_PATH.parent.mkdir(parents=True, exist_ok=True)
tableau_df.to_csv(TABLEAU_READY_PATH, index=False)
print(f'✓ Saved Tableau-ready dataset → {TABLEAU_READY_PATH}')
print(f'  Size on disk: {TABLEAU_READY_PATH.stat().st_size / 1e6:.1f} MB')

## Output summary

Files written to `data/processed/`:

- `cleaned_dataset.csv` — row-level cleaned output (from notebook 02)
- `tableau_ready_dataset.csv` — trimmed row-level file for Tableau
- `kpi/kpi_state.csv`
- `kpi/kpi_city_top100.csv`
- `kpi/kpi_weather.csv`
- `kpi/kpi_rush_hour.csv`
- `kpi/kpi_poi.csv`
- `kpi/kpi_yoy.csv`
- `kpi/kpi_day_night.csv`

All KPI files can be connected in Tableau as supplementary data sources for the executive-view sheets, while `tableau_ready_dataset.csv` drives operational drill-downs and map views.